# VSHORAD Embedded — TensorRT Export

**Tier**: Embedded (Jetson / edge GPU)

| Component | Source | Export | Precision |
|-----------|--------|--------|-----------|
| YOLO | YOLOv8m (.pt) | TensorRT | FP16 / INT8 |
| Swin | Swin-Small (.pth) | ONNX → TensorRT | FP16 |

Exports Tactical-tier models to TensorRT for edge deployment.
Resolution reduced to 640px (YOLO) for maximum throughput.

## 1 Setup

In [ ]:
# 1.1 Installation

print(" Installing TensorRT and dependencies...")
!pip uninstall numpy -y -q
!pip install numpy==1.26.4 -q
!pip install ultralytics==8.3.0 timm==1.0.11 -q
!pip install onnx onnxruntime-gpu -q
!pip uninstall onnx onnxruntime onnxruntime-gpu -y
!pip install onnx==1.16.0 onnxruntime-gpu==1.18.0 onnxscript onnxslim==0.1.34

# TensorRT is pre-installed on Colab z GPU
print(" Installation complete!")

In [ ]:
# 1.2 Verify GPU and TensorRT

import torch
import subprocess

print(" DIAGNOSTICS:\n")

# GPU
if torch.cuda.is_available():
  gpu_name = torch.cuda.get_device_name(0)
  vram = torch.cuda.get_device_properties(0).total_memory / 1e9
  print(f" GPU: {gpu_name}")
  print(f"  VRAM: {vram:.1f} GB")
else:
  print(" No GPU available! TensorRT wymaga GPU.")
  raise RuntimeError("Brak GPU")

# TensorRT
try:
  import tensorrt as trt
  print(f" TensorRT: {trt.__version__}")
except ImportError:
  print(" TensorRT not installed directly, but ultralytics will handle it")

# CUDA
print(f" CUDA: {torch.version.cuda}")
print(f" PyTorch: {torch.__version__}")

In [ ]:
# 1.3 Imports and paths

import os, sys, json, shutil, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import timm

from ultralytics import YOLO

# Mount Drive
try:
  from google.colab import drive
  drive.mount('/content/drive')
  IN_COLAB = True
except:
  IN_COLAB = False

DRIVE_ROOT = Path('/content/drive/MyDrive/Inżynierka') if IN_COLAB else Path('./Inzynierka')

# Source - trained models
MEDIUM_ROOT = DRIVE_ROOT / 'Medium_components'
YOLO_SOURCE = MEDIUM_ROOT / 'checkpoints' / 'yolo' / 'medium_yolov8m_960' / 'weights' / 'best.pt'
SWIN_SOURCE = MEDIUM_ROOT / 'checkpoints' / 'swin' / 'medium_swin_small_224' / 'best.pth'

# Cel - Small/Embedded
OUTPUT_ROOT = DRIVE_ROOT / 'Small_components'
YOLO_EXPORT_DIR = OUTPUT_ROOT / 'exports' / 'yolo' / 'yolov8m_640_tensorrt'
SWIN_EXPORT_DIR = OUTPUT_ROOT / 'exports' / 'swin' / 'swin_small_224_tensorrt'

# Create directories
for d in [YOLO_EXPORT_DIR, SWIN_EXPORT_DIR]:
  d.mkdir(parents=True, exist_ok=True)

print(f" YOLO source: {YOLO_SOURCE}")
print(f"  Exists: {YOLO_SOURCE.exists()}")
print(f" Swin source: {SWIN_SOURCE}")
print(f"  Exists: {SWIN_SOURCE.exists()}")
print(f"\n Output YOLO: {YOLO_EXPORT_DIR}")
print(f" Output Swin: {SWIN_EXPORT_DIR}")

## 2 YOLO Export to TensorRT

In [ ]:
# 2.1 Export YOLO to TensorRT FP16

print(" Exporting YOLO to TensorRT FP16...")
print(f"  Input: {YOLO_SOURCE}")
print(f"  Output size: 640px")
print()

# Load model
yolo = YOLO(str(YOLO_SOURCE))

# Export to TensorRT FP16
t0 = time.time()
yolo.export(
  format='engine',
  imgsz=640,
  half=True,      # FP16
  int8=False,
  device=0,
  workspace=4,     # GB
  verbose=True,
)
print(f"\n Export time: {time.time()-t0:.1f}s")

# Move file
engine_src = YOLO_SOURCE.parent / 'best.engine'
engine_dst = YOLO_EXPORT_DIR / 'yolov8m_640_fp16.engine'

if engine_src.exists():
  shutil.copy(str(engine_src), str(engine_dst))
  size_mb = engine_dst.stat().st_size / 1e6
  print(f"\n Saved: {engine_dst}")
  print(f"  Rozmiar: {size_mb:.1f} MB")
else:
  print(f" Nie znaleziono: {engine_src}")
  # Check location
  !find /content -name "*.engine" 2>/dev/null | head -5

In [ ]:
# 2.2 Export YOLO to TensorRT INT8 (requires calibration)

print(" Exporting YOLO to TensorRT INT8...")
print("  INT8 requires calibration data - using defaults\n")

# Need calibration data for INT8
# Ultralytics will use COCO samples if none provided

# Reload model (clean state)
yolo_int8 = YOLO(str(YOLO_SOURCE))

t0 = time.time()
try:
  yolo_int8.export(
    format='engine',
    imgsz=640,
    half=False,     # Nie FP16
    int8=True,      # INT8!
    device=0,
    workspace=4,
    verbose=True,
    # data='coco128.yaml', # Optional: custom calibration data
  )
  print(f"\n Export time: {time.time()-t0:.1f}s")

  # Move file
  engine_src = YOLO_SOURCE.parent / 'best.engine'
  engine_dst = YOLO_EXPORT_DIR / 'yolov8m_640_int8.engine'

  if engine_src.exists():
    shutil.copy(str(engine_src), str(engine_dst))
    size_mb = engine_dst.stat().st_size / 1e6
    print(f"\n Saved: {engine_dst}")
    print(f"  Rozmiar: {size_mb:.1f} MB")
except Exception as e:
  print(f" INT8 export error: {e}")
  print("  INT8 may require custom calibration data")
  print("  Use FP16 as fallback")

## 3 Swin Export to TensorRT

Two-step process: PyTorch → ONNX → TensorRT.

In [ ]:
# 3.1 Load Swin and export to ONNX

print(" Exporting Swin to ONNX (intermediate step)...")

SWIN_CLASSES = [
  'AH-1', 'AH-64', 'AN-124', 'AN-26', 'A-10', 'AV-8B', 'B1', 'B2', 'B52',
  'C130', 'C17', 'C5', 'CH-47', 'E2', 'E3', 'E7', 'EUROFIGHTER',
  'F-117', 'F-14', 'F-15', 'F-16', 'F-22', 'F-35', 'F-4', 'FA-18',
  'Flanker_family', 'Fulcrum_family', 'GRIPEN', 'H6', 'IL-76',
  'J-10', 'J-20', 'J-31', 'JF-17', 'KA-52', 'KC-135', 'MI-24', 'MI-28',
  'MI-8', 'MIG-31', 'MIRAGE-2000', 'RAFALE', 'SR-71', 'SU-25', 'SU-34',
  'SU-57', 'TIGER', 'TU-160', 'TU-95', 'U2', 'UH60M', 'V22', 'VULCAN',
  'Y20', 'YF-23', 'Z10'
]
NUM_CLASSES = len(SWIN_CLASSES)
IMG_SIZE = 224

# Load model
swin = timm.create_model('swin_small_patch4_window7_224', pretrained=False, num_classes=NUM_CLASSES)

# Load weights
checkpoint = torch.load(str(SWIN_SOURCE), map_location='cuda')
state_dict = (checkpoint.get('ema_state_dict') or checkpoint.get('ema') or
       checkpoint.get('model_state_dict') or checkpoint.get('model') or checkpoint)
swin.load_state_dict(state_dict)
swin.eval()
swin.cuda()

print(f" Swin loaded: {NUM_CLASSES} classes")

# Export to ONNX
onnx_path = SWIN_EXPORT_DIR / 'swin_small_224_fp32.onnx'
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).cuda()

torch.onnx.export(
  swin,
  dummy_input,
  str(onnx_path),
  export_params=True,
  opset_version=17,
  do_constant_folding=True,
  input_names=['input'],
  output_names=['output'],
  dynamic_axes=None, # Fixed size for TensorRT
)

size_mb = onnx_path.stat().st_size / 1e6
print(f" ONNX saved: {onnx_path}")
print(f"  Rozmiar: {size_mb:.1f} MB")

In [ ]:
# 3.1c Merge ONNX with external weights

import onnx
from onnx.external_data_helper import convert_model_to_external_data, load_external_data_for_model

print(" Merging ONNX with external weights...")

onnx_path = SWIN_EXPORT_DIR / 'swin_small_224_fp32.onnx'
onnx_merged_path = SWIN_EXPORT_DIR / 'swin_small_224_fp32_merged.onnx'

# Load model with external data
model = onnx.load(str(onnx_path), load_external_data=True)

# Save with embedded weights
onnx.save(model, str(onnx_merged_path), save_as_external_data=False)

size_mb = onnx_merged_path.stat().st_size / 1e6
print(f" ONNX merged: {onnx_merged_path}")
print(f"  Rozmiar: {size_mb:.1f} MB")

In [ ]:
# 3.2c Convert merged ONNX to TensorRT FP16

import tensorrt as trt

print(" Converting Swin ONNX → TensorRT FP16...")

onnx_path = str(SWIN_EXPORT_DIR / 'swin_small_224_fp32_merged.onnx')
engine_path = str(SWIN_EXPORT_DIR / 'swin_small_224_fp16.engine')

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

builder = trt.Builder(TRT_LOGGER)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, TRT_LOGGER)

with open(onnx_path, 'rb') as f:
  if not parser.parse(f.read()):
    for i in range(parser.num_errors):
      print(f"  {parser.get_error(i)}")
    raise RuntimeError("ONNX parsing failed")

print(f"  ONNX parsed: {network.num_layers} layers")

config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
config.set_flag(trt.BuilderFlag.FP16)

print("  ⏳ Building engine (2-5 min)...")
t0 = time.time()
serialized_engine = builder.build_serialized_network(network, config)
print(f"   Build: {time.time()-t0:.1f}s")

if serialized_engine:
  with open(engine_path, 'wb') as f:
    f.write(serialized_engine)
  size_mb = Path(engine_path).stat().st_size / 1e6
  print(f"\n TensorRT FP16: {engine_path}")
  print(f"  Rozmiar: {size_mb:.1f} MB")
else:
  print(" Build failed - use ONNX Runtime as fallback")

## 4 Verification & Benchmark

In [ ]:
# 4.1 List exported files

print(" Exported models:\n")

def list_exports(path):
  if not path.exists():
    print(f"  {path} does not exist")
    return
  for f in sorted(path.glob('*')):
    if f.is_file():
      size_mb = f.stat().st_size / 1e6
      print(f"  {'' if f.suffix in ['.engine', '.onnx', '.ts'] else ''} {f.name} ({size_mb:.1f} MB)")

print("YOLO:")
list_exports(YOLO_EXPORT_DIR)

print("\nSwin:")
list_exports(SWIN_EXPORT_DIR)

In [ ]:
# 4.2 BENCHMARK YOLO TENSORRT

print(" Benchmark YOLO TensorRT FP16...\n")

engine_path = YOLO_EXPORT_DIR / 'yolov8m_640_fp16.engine'

if engine_path.exists():
  yolo_trt = YOLO(str(engine_path))

  # Warmup
  dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
  for _ in range(10):
    yolo_trt(dummy, verbose=False)

  # Benchmark
  times = []
  for _ in range(100):
    t0 = time.perf_counter()
    yolo_trt(dummy, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)

  print(f"  Mean latency: {np.mean(times):.2f} ms")
  print(f"  Std: {np.std(times):.2f} ms")
  print(f"  FPS: {1000/np.mean(times):.1f}")
else:
  print(f" Nie znaleziono: {engine_path}")

In [ ]:
# 4.3 Compare: PyTorch vs ONNX vs TensorRT

print(" YOLO performance comparison:\n")

results = {}
dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# PyTorch
print("Testing PyTorch (.pt)...")
yolo_pt = YOLO(str(YOLO_SOURCE))
for _ in range(5): yolo_pt(dummy, verbose=False) # warmup
times = []
for _ in range(50):
  t0 = time.perf_counter()
  yolo_pt(dummy, verbose=False)
  times.append((time.perf_counter() - t0) * 1000)
results['PyTorch'] = np.mean(times)
print(f"  PyTorch: {results['PyTorch']:.2f} ms ({1000/results['PyTorch']:.1f} FPS)")

# TensorRT FP16
trt_fp16 = YOLO_EXPORT_DIR / 'yolov8m_640_fp16.engine'
if trt_fp16.exists():
  print("Testing TensorRT FP16...")
  yolo_trt16 = YOLO(str(trt_fp16))
  for _ in range(5): yolo_trt16(dummy, verbose=False)
  times = []
  for _ in range(50):
    t0 = time.perf_counter()
    yolo_trt16(dummy, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)
  results['TensorRT FP16'] = np.mean(times)
  print(f"  TensorRT FP16: {results['TensorRT FP16']:.2f} ms ({1000/results['TensorRT FP16']:.1f} FPS)")

# TensorRT INT8
trt_int8 = YOLO_EXPORT_DIR / 'yolov8m_640_int8.engine'
if trt_int8.exists():
  print("Testing TensorRT INT8...")
  yolo_trt8 = YOLO(str(trt_int8))
  for _ in range(5): yolo_trt8(dummy, verbose=False)
  times = []
  for _ in range(50):
    t0 = time.perf_counter()
    yolo_trt8(dummy, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)
  results['TensorRT INT8'] = np.mean(times)
  print(f"  TensorRT INT8: {results['TensorRT INT8']:.2f} ms ({1000/results['TensorRT INT8']:.1f} FPS)")

# Summary
print("\n" + "="*50)
print(" SUMMARY:")
print("="*50)
baseline = results.get('PyTorch', 1)
for name, ms in sorted(results.items(), key=lambda x: x[1]):
  speedup = baseline / ms
  print(f"  {name:<20} {ms:>6.2f} ms ({1000/ms:>5.1f} FPS) {speedup:>4.1f}x")

In [ ]:
# 4.4 Save metadata

import json
from datetime import datetime

metadata = {
  'export_date': datetime.now().isoformat(),
  'source': {
    'yolo': str(YOLO_SOURCE),
    'swin': str(SWIN_SOURCE),
  },
  'yolo_config': {
    'model': 'yolov8m',
    'imgsz': 640,
    'formats': ['fp16.engine', 'int8.engine'],
  },
  'swin_config': {
    'model': 'swin_small_patch4_window7_224',
    'img_size': 224,
    'num_classes': 56,
    'formats': ['fp32.onnx', 'fp16.engine'],
  },
  'gpu_used': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
  'benchmark_results': results if 'results' in dir() else {},
}

meta_path = OUTPUT_ROOT / 'metadata_tensorrt.json'
with open(meta_path, 'w') as f:
  json.dump(metadata, f, indent=2)

print(f" Metadata saved: {meta_path}")

## Export Complete

Models exported to TensorRT. Use `run.py --tier embedded` to run inference.